In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.chdir("..")  # change working directory to project root
%load_ext autoreload
%autoreload 2

In [3]:
from src.db.client import db

In [4]:
query = """
select * from assets;
"""
assets = db.query_df(query)

2026-03-14 15:13:33 [INFO] Opening DuckDB at data/quant.db
2026-03-14 15:13:33 [INFO] DB schema up to date (version 7)
2026-03-14 15:13:33 [INFO] DuckDB connection closed


In [35]:
assets

,id,ticker,name,asset_class,exchange,currency,is_active,created_at
0,1,AAPL,Apple Inc.,equity,None,USD,True,2026-02-24 21:27:48.037848-05:00
1,2,MMM,3M,equity,None,USD,True,2026-02-28 12:28:55.715840-05:00
2,3,AOS,A. O. Smith,equity,None,USD,True,2026-02-28 12:28:55.718856-05:00
3,4,ABT,Abbott Laboratories,equity,None,USD,True,2026-02-28 12:28:55.719981-05:00
4,5,ABBV,AbbVie,equity,None,USD,True,2026-02-28 12:28:55.721003-05:00
...,...,...,...,...,...,...,...,...
544,100011,000001.SS,Shanghai Composite,equity,None,USD,True,2026-03-14 14:17:56.315287-04:00
545,100012,^KS11,Kospi,equity,None,USD,True,2026-03-14 14:17:56.315287-04:00
546,100013,^GSPTSE,TSX Composite,equity,None,USD,True,2026-03-14 14:17:56.315287-04:00
547,100014,^AXJO,ASX 200,equity,None,USD,True,2026-03-14 14:17:56.315287-04:00


In [27]:
asset_ids = assets.loc[
    (assets['id'] > 100000) & (assets['id'] < 200000),
    'id'
].tolist()

In [59]:
query = f"""
select * from prices where asset_id in {tuple(asset_ids)}
"""
levels = db.query_df(query)

2026-03-14 15:47:09 [INFO] Opening DuckDB at data/quant.db
2026-03-14 15:47:09 [INFO] DB schema up to date (version 7)
2026-03-14 15:47:10 [INFO] DuckDB connection closed


In [62]:
asset_mapping = assets[['id','ticker']].set_index('id').to_dict()['ticker']

In [63]:
levels['timestamp'] = levels['timestamp'].dt.date

In [68]:
asset_returns = levels.pivot(index = 'timestamp', columns = 'asset_id', values = 'close')
asset_returns.columns = asset_returns.columns.map(asset_mapping)
asset_returns = np.log(asset_returns) - np.log(asset_returns.shift(1))

In [69]:
asset_returns

asset_id,^GSPC,^NDX,^RUT,^STOXX50E,^FTSE,^GDAXI,^FCHI,^IBEX,^N225,^HSI,000001.SS,^KS11,^GSPTSE,^AXJO,^BVSP
timestamp,,,,,,,,,,,,,,,
1970-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-05,0.004934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-06,-0.006871,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001401,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-07,-0.002049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.004691,NaN,NaN,NaN,NaN,NaN,NaN
1970-01-08,0.000540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.047578,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-09,0.008270,0.013071,0.011172,-0.006085,-0.003438,-0.007730,-0.009822,-0.008599,-0.053398,-0.013635,-0.006712,-0.061477,0.003187,-0.028885,0.008604
2026-03-10,-0.002137,-0.000432,-0.002191,0.026380,0.015749,0.023610,0.017781,0.030072,0.028413,0.021471,0.006456,0.052072,0.002450,0.010826,0.013898
2026-03-11,-0.000838,0.000342,-0.002035,-0.007306,-0.005625,-0.013804,-0.001932,-0.005351,0.014221,-0.002358,0.002494,0.013886,-0.004546,0.005839,0.002841
